## Diffusion-Planner: mini split debug + visualization

This notebook helps you:

- Run **1-scenario** closed-loop simulation on nuPlan **mini** DB
- Inspect **model inputs / outputs** for one timestep
- Visualize **BEV map + agent boxes + predicted trajectory**
- Export an **MP4 video** over the whole scenario

### Assumptions
- You run this notebook **inside the Docker container** where you already installed:
  - `nuplan-devkit`
  - this repo (`pip install -e .`)
- Your mini DB exists at `data/nuplan_data/nuplan-v1.1/splits/mini/...db`
- Your maps exist at `data/nuplan_data/maps`



In [ ]:
import os
from pathlib import Path

# Edit these paths if needed
NUPLAN_DEVKIT_ROOT = Path(os.environ.get("NUPLAN_DEVKIT_ROOT", "/opt/nuplan-devkit"))
NUPLAN_DATA_ROOT = Path(os.environ.get("NUPLAN_DATA_ROOT", "/workspace/data/nuplan_data"))
NUPLAN_MAPS_ROOT = Path(os.environ.get("NUPLAN_MAPS_ROOT", str(NUPLAN_DATA_ROOT / "maps")))
NUPLAN_EXP_ROOT = Path(os.environ.get("NUPLAN_EXP_ROOT", "/workspace/data/exp"))

DB_FILE = NUPLAN_DATA_ROOT / "nuplan-v1.1/splits/mini/2021.05.12.22.00.38_veh-35_01008_01518.db"
MAP_VERSION = "nuplan-maps-v1.0"
SENSOR_ROOT = NUPLAN_DATA_ROOT / "nuplan-v1.1/sensor_blobs"

CHECKPOINT_ARGS = Path("/workspace/checkpoints/args.json")
CHECKPOINT_PTH = Path("/workspace/checkpoints/model.pth")

print("NUPLAN_DEVKIT_ROOT:", NUPLAN_DEVKIT_ROOT)
print("NUPLAN_DATA_ROOT:", NUPLAN_DATA_ROOT)
print("NUPLAN_MAPS_ROOT:", NUPLAN_MAPS_ROOT)
print("NUPLAN_EXP_ROOT:", NUPLAN_EXP_ROOT)
print("DB_FILE:", DB_FILE)
print("CHECKPOINT_ARGS:", CHECKPOINT_ARGS)
print("CHECKPOINT_PTH:", CHECKPOINT_PTH)

assert (NUPLAN_DEVKIT_ROOT / "nuplan/planning/script/run_simulation.py").exists(), "nuplan-devkit not found"
assert DB_FILE.exists(), "mini DB not found"
assert NUPLAN_MAPS_ROOT.exists(), "maps root not found"
assert CHECKPOINT_ARGS.exists(), "args.json not found"
assert CHECKPOINT_PTH.exists(), "model.pth not found"

In [ ]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "imageio", "imageio-ffmpeg", "pandas"])

In [ ]:
# Run a 1-scenario simulation on the mini DB (sequential worker to ensure SimulationLog is written)

import os
import sys
import subprocess
from datetime import datetime

os.environ["HYDRA_FULL_ERROR"] = "1"
os.environ["NUPLAN_DEVKIT_ROOT"] = str(NUPLAN_DEVKIT_ROOT)
os.environ["NUPLAN_DATA_ROOT"] = str(NUPLAN_DATA_ROOT)
os.environ["NUPLAN_MAPS_ROOT"] = str(NUPLAN_MAPS_ROOT)
os.environ["NUPLAN_EXP_ROOT"] = str(NUPLAN_EXP_ROOT)

run_sim = NUPLAN_DEVKIT_ROOT / "nuplan/planning/script/run_simulation.py"

# Unique output run folder
stamp = datetime.now().strftime("%Y.%m.%d.%H.%M.%S")
job_uid = f"diffusion_planner/mini_one/{stamp}"

cmd = [
    sys.executable, str(run_sim),
    "+simulation=closed_loop_nonreactive_agents",
    "planner=diffusion_planner",
    f"planner.diffusion_planner.config.args_file={CHECKPOINT_ARGS}",
    f"planner.diffusion_planner.ckpt_path={CHECKPOINT_PTH}",
    "scenario_builder=nuplan",
    f"scenario_builder.data_root={DB_FILE.parent}",
    f"scenario_builder.db_files=[{DB_FILE}]",
    "scenario_filter=test_one_scenatio",
    "scenario_filter.scenario_tokens=null",
    "scenario_filter.scenario_types=null",
    "scenario_filter.log_names=null",
    "scenario_filter.num_scenarios_per_type=null",
    "scenario_filter.limit_total_scenarios=1",
    "scenario_filter.shuffle=false",
    "worker=sequential",
    "distributed_mode=SINGLE_NODE",
    f"experiment_uid={job_uid}",
    "hydra.searchpath=[pkg://diffusion_planner.config.scenario_filter, pkg://diffusion_planner.config, pkg://nuplan.planning.script.config.common, pkg://nuplan.planning.script.experiments]",
]

print(" ".join(cmd))
subprocess.check_call(cmd)

# nuPlan outputs here:
RUN_DIR = NUPLAN_EXP_ROOT / "exp/simulation/closed_loop_nonreactive_agents" / stamp
print("RUN_DIR:", RUN_DIR)
print("(If the folder name differs, locate the newest folder in that parent directory.)")

In [ ]:
# Find the newest run directory
import glob

parent = NUPLAN_EXP_ROOT / "exp/simulation/closed_loop_nonreactive_agents"
runs = sorted(parent.glob("*"), key=lambda p: p.stat().st_mtime)
assert runs, f"No runs found under {parent}"
RUN_DIR = runs[-1]
print("Using RUN_DIR:", RUN_DIR)

In [ ]:
# Load SimulationLog (robust: works if RUN_DIR is the run folder OR a parent folder)
import glob
from nuplan.planning.simulation.simulation_log import SimulationLog

# If user accidentally points RUN_DIR at the simulation_log folder, go one level up
if RUN_DIR.name == "simulation_log":
    RUN_DIR = RUN_DIR.parent

# Search in common locations first
candidates = [
    RUN_DIR / "simulation_log",
    RUN_DIR,
]

log_files = []
for base in candidates:
    log_files.extend(glob.glob(str(base / "**" / "*.msgpack.xz"), recursive=True))
    log_files.extend(glob.glob(str(base / "**" / "*.pkl.xz"), recursive=True))

# De-dup
log_files = sorted(set(log_files))
print("RUN_DIR:", RUN_DIR)
print("SimulationLog files:", len(log_files))

if not log_files:
    # Helpful hint: show what folders exist
    print("Contents:", [p.name for p in RUN_DIR.iterdir()])
    raise AssertionError(
        f"No SimulationLog found under {RUN_DIR}. Expected something like {RUN_DIR}/simulation_log/**/<token>.msgpack.xz"
    )

SIM_LOG = SimulationLog.load_data(Path(log_files[0]))
HIST = SIM_LOG.simulation_history
print("Loaded:", log_files[0])
print("scenario:", SIM_LOG.scenario.log_name, SIM_LOG.scenario.scenario_name, SIM_LOG.scenario.scenario_type)
print("steps:", len(HIST.data))

In [ ]:
# Visualize a single BEV frame: map + agents + ego + predicted trajectory (all aligned from SimulationLog)

import matplotlib.pyplot as plt
from nuplan.common.actor_state.state_representation import Point2D
from nuplan.common.maps.maps_datatypes import SemanticMapLayer

step = min(40, len(HIST.data) - 1)
radius = 80.0

sample = HIST.data[step]
map_api = HIST.map_api

ego = sample.ego_state
cx, cy = ego.rear_axle.x, ego.rear_axle.y

plt.figure(figsize=(9, 9))

# map
layers = [SemanticMapLayer.LANE, SemanticMapLayer.LANE_CONNECTOR]
objs = map_api.get_proximal_map_objects(Point2D(cx, cy), radius, layers)
for items in objs.values():
    for obj in items:
        poly = getattr(obj, "polygon", None)
        if poly is None:
            continue
        x, y = poly.exterior.xy
        plt.plot(x, y, linewidth=0.6, alpha=0.6, color="gray")

# agents (boxes)
tracked = getattr(sample.observation, "tracked_objects", None)
if tracked is not None:
    for o in tracked.tracked_objects:
        b = getattr(o, "box", None) or getattr(o, "oriented_box", None)
        if b is None:
            continue
        corners = b.all_corners()
        xs = [p.x for p in corners] + [corners[0].x]
        ys = [p.y for p in corners] + [corners[0].y]
        plt.plot(xs, ys, linewidth=1.0, color="dodgerblue", alpha=0.9)

# ego (box)
ego_box = ego.car_footprint.oriented_box
corners = ego_box.all_corners()
xs = [p.x for p in corners] + [corners[0].x]
ys = [p.y for p in corners] + [corners[0].y]
plt.plot(xs, ys, linewidth=2.0, color="red", label="ego")

# predicted trajectory
traj = sample.trajectory
states = traj.get_sampled_trajectory()
px = [s.rear_axle.x for s in states]
py = [s.rear_axle.y for s in states]
plt.plot(px, py, linewidth=2.5, color="gold", label="predicted")

plt.axis("equal")
plt.xlim(cx - radius, cx + radius)
plt.ylim(cy - radius, cy + radius)
plt.grid(True, alpha=0.3)
plt.legend()
plt.title(f"BEV frame step={step}\n{SIM_LOG.scenario.log_name} / {SIM_LOG.scenario.scenario_name}")

out = RUN_DIR / "bev_frame.png"
plt.savefig(out, dpi=200, bbox_inches="tight")
print("Saved:", out)
plt.show()

In [ ]:
# Render an MP4 over the whole scenario

import numpy as np
import imageio.v2 as imageio

fps = 10
radius = 80.0
layers = [SemanticMapLayer.LANE, SemanticMapLayer.LANE_CONNECTOR]

writer = imageio.get_writer(str(RUN_DIR / "bev_whole_scene.mp4"), fps=fps, codec="libx264", quality=8)

fig, ax = plt.subplots(figsize=(7, 7), dpi=140)

for k, sample in enumerate(HIST.data):
    ax.clear()
    ego = sample.ego_state
    cx, cy = ego.rear_axle.x, ego.rear_axle.y

    # map patch
    objs = HIST.map_api.get_proximal_map_objects(Point2D(cx, cy), radius, layers)
    for items in objs.values():
        for obj in items:
            poly = getattr(obj, "polygon", None)
            if poly is None:
                continue
            x, y = poly.exterior.xy
            ax.plot(x, y, linewidth=0.6, alpha=0.6, color="gray")

    # agents
    tracked = getattr(sample.observation, "tracked_objects", None)
    if tracked is not None:
        for o in tracked.tracked_objects:
            b = getattr(o, "box", None) or getattr(o, "oriented_box", None)
            if b is None:
                continue
            corners = b.all_corners()
            xs = [p.x for p in corners] + [corners[0].x]
            ys = [p.y for p in corners] + [corners[0].y]
            ax.plot(xs, ys, linewidth=1.0, color="dodgerblue", alpha=0.9)

    # ego box
    ego_box = ego.car_footprint.oriented_box
    corners = ego_box.all_corners()
    xs = [p.x for p in corners] + [corners[0].x]
    ys = [p.y for p in corners] + [corners[0].y]
    ax.plot(xs, ys, linewidth=2.0, color="red")

    # predicted trajectory
    traj = sample.trajectory
    states = traj.get_sampled_trajectory()
    px = [s.rear_axle.x for s in states]
    py = [s.rear_axle.y for s in states]
    ax.plot(px, py, linewidth=2.5, color="gold")

    ax.set_aspect("equal", "box")
    ax.set_xlim(cx - radius, cx + radius)
    ax.set_ylim(cy - radius, cy + radius)
    ax.grid(True, alpha=0.3)
    ax.set_title(f"step={k}")

    fig.canvas.draw()
    frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    writer.append_data(frame)

writer.close()
plt.close(fig)
print("Saved:", RUN_DIR / "bev_whole_scene.mp4")

In [ ]:
# Inspect Diffusion-Planner model input/output for ONE timestep
# This uses the same nuPlan history buffer interface as the planner.

import torch
from nuplan.planning.simulation.history.simulation_history_buffer import SimulationHistoryBuffer
from nuplan.planning.simulation.observation.observation_type import DetectionsTracks

from diffusion_planner.utils.config import Config
from diffusion_planner.model.diffusion_planner import Diffusion_Planner
from diffusion_planner.data_process.data_processor import DataProcessor

# Build a Scenario again (only for creating a history buffer + map_api + route ids)
from nuplan.planning.scenario_builder.nuplan_db.nuplan_scenario_builder import NuPlanScenarioBuilder
from nuplan.planning.scenario_builder.scenario_filter import ScenarioFilter
from nuplan.planning.utils.multithreading.worker_sequential import Sequential

worker = Sequential()
sf = ScenarioFilter(
    scenario_types=None, scenario_tokens=[SIM_LOG.scenario.scenario_name], log_names=None, map_names=None,
    num_scenarios_per_type=None, limit_total_scenarios=1,
    timestamp_threshold_s=0,
    ego_displacement_minimum_m=None, ego_start_speed_threshold=None, ego_stop_speed_threshold=None,
    speed_noise_tolerance=None,
    expand_scenarios=False, remove_invalid_goals=True, shuffle=False,
)

builder = NuPlanScenarioBuilder(
    data_root=str(DB_FILE.parent),
    map_root=str(NUPLAN_MAPS_ROOT),
    sensor_root=str(SENSOR_ROOT),
    db_files=[str(DB_FILE)],
    map_version=MAP_VERSION,
    include_cameras=False,
)
scenario = builder.get_scenarios(sf, worker)[0]

# Build history buffer at iteration `it`
it = 40
buffer_size = 21  # matches DataProcessor slicing neighbor_agents_past[:, -21:]

# We create the rolling buffer by querying past ego + past tracked objects up to this iteration.
buffer_duration = buffer_size * scenario.database_interval
past_obs = list(scenario.get_past_tracked_objects(iteration=it, time_horizon=buffer_duration, num_samples=buffer_size))
past_ego = list(scenario.get_ego_past_trajectory(iteration=it, time_horizon=buffer_duration, num_samples=buffer_size))

history_buffer = SimulationHistoryBuffer.initialize_from_list(
    buffer_size=buffer_size,
    ego_states=past_ego,
    observations=past_obs,
    sample_interval=scenario.database_interval,
)

traffic_light_data = list(scenario.get_traffic_light_status_at_iteration(it))
route_roadblock_ids = scenario.get_route_roadblock_ids()
map_api = scenario.map_api

# Load model + config
cfg = Config(args_file=str(CHECKPOINT_ARGS), guidance_fn=None)
model = Diffusion_Planner(cfg)
state = torch.load(str(CHECKPOINT_PTH), map_location="cpu")
state = state.get("ema_state_dict", state)
state = state.get("model", state)
model_state = {k[len("module."):]: v for k, v in state.items() if k.startswith("module.")}
model.load_state_dict(model_state)
model.eval()

processor = DataProcessor(cfg)
inputs = processor.observation_adapter(history_buffer, traffic_light_data, map_api, route_roadblock_ids, device="cpu")

print("Model input keys:", sorted(inputs.keys()))
for k, v in inputs.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {tuple(v.shape)} {v.dtype} {v.device}")

with torch.no_grad():
    enc_out, dec_out = model(inputs)

print("\nEncoder output keys:", sorted(enc_out.keys()))
for k, v in enc_out.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {tuple(v.shape)}")

print("\nDecoder output keys:", sorted(dec_out.keys()))
for k, v in dec_out.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {tuple(v.shape)}")

# The main predicted trajectory is usually in dec_out['prediction']
if "prediction" in dec_out:
    pred = dec_out["prediction"]
    print("\nprediction sample:", pred[0, 0, :5])

## Tutorial: how raw nuPlan data becomes model input

At inference time, Diffusion-Planner consumes a dictionary of tensors produced by:

`nuPlan Scenario` → `SimulationHistoryBuffer` → `DataProcessor.observation_adapter(...)` → `convert_to_model_inputs(...)` → **Encoder / Decoder**

### Key idea (coordinate system)
All dynamic & map inputs are converted into an **ego-centric local frame** anchored at the current ego rear axle pose:

- anchor pose: \([$x_0$, $y_0$, $\psi_0$] \)
- everything becomes local: \($x'$\), \($y'$\), and headings become \($\cos$($\Delta\psi$), \sin($\Delta\psi$)\)

This is why you see many tensors with \(x,y,$\cos$,$\sin$\) instead of \(x,y,$\psi$\).

### What each input key means (high level)
- `neighbor_agents_past`: past trajectories of nearby agents in ego frame.
- `static_objects`: static obstacles (barriers/cones/etc) near ego.
- `lanes`, `lanes_speed_limit`, `lanes_has_speed_limit`: vectorized map context around ego.
- `route_lanes`: the route polylines used by the decoder's `RouteEncoder`.
- `ego_current_state`: ego state at current step (the decoder only uses the first 4 dims).

The next cell prints the *intermediate* numpy arrays (before tensor conversion) so you can trace **initial → processed**.


In [ ]:
# Inspect intermediate preprocessing outputs (raw -> processed -> tensors)
# This mirrors DataProcessor.observation_adapter, but prints each stage.

import numpy as np
import torch

from diffusion_planner.data_process.agent_process import (
    sampled_tracked_objects_to_array_list,
    sampled_static_objects_to_array_list,
    agent_past_process,
)
from diffusion_planner.data_process.map_process import (
    get_neighbor_vector_set_map,
    map_process,
)
from diffusion_planner.data_process.roadblock_utils import route_roadblock_correction
from diffusion_planner.data_process.utils import convert_to_model_inputs
from nuplan.common.actor_state.state_representation import Point2D

# pick an iteration to debug
it_dbg = 40

# 1) raw ego + anchor
ego_state = scenario.get_ego_state_at_iteration(it_dbg)
anchor_ego_state = np.array([ego_state.rear_axle.x, ego_state.rear_axle.y, ego_state.rear_axle.heading], dtype=np.float64)
ego_coords = Point2D(ego_state.rear_axle.x, ego_state.rear_axle.y)
print("anchor_ego_state [x,y,heading]:", anchor_ego_state)

# 2) raw observation buffer (past + current)
buffer_size = 21
buffer_duration = buffer_size * scenario.database_interval
observation_buffer = list(scenario.get_past_tracked_objects(iteration=it_dbg, time_horizon=buffer_duration, num_samples=buffer_size))
print("observation_buffer len:", len(observation_buffer), "type[0]:", type(observation_buffer[0]))

# 3) arrayify agents / static
neighbor_agents_past_list, neighbor_agents_types_list = sampled_tracked_objects_to_array_list(observation_buffer)
static_objects_raw, static_objects_types = sampled_static_objects_to_array_list(observation_buffer[-1])
print("raw neighbor frames:", len(neighbor_agents_past_list))
print("raw neighbor frame[0] shape:", neighbor_agents_past_list[0].shape, "(AgentInternalIndex.dim)")
print("raw static_objects shape:", static_objects_raw.shape)

# 4) agent selection, padding, ego-centric transform
#    (ego_agent_past is None for inference)
ego_agent_past = None
cfg_dbg = cfg  # from the model IO cell

ego_agent_past, neighbor_agents_past, neighbor_indices, static_objects = agent_past_process(
    ego_agent_past,
    neighbor_agents_past_list,
    neighbor_agents_types_list,
    num_agents=cfg_dbg.agent_num,
    static_objects=static_objects_raw,
    static_objects_types=static_objects_types,
    num_static=cfg_dbg.static_objects_num,
    max_ped_bike=10,
    anchor_ego_state=anchor_ego_state,
)
print("processed neighbor_agents_past shape:", neighbor_agents_past.shape, "(T, N, D or N,T,...) depending on function)")
print("processed static_objects shape:", static_objects.shape)

# 5) map vectorization
traffic_light_data = list(scenario.get_traffic_light_status_at_iteration(it_dbg))
route_roadblock_ids = scenario.get_route_roadblock_ids()
route_roadblock_ids = route_roadblock_correction(ego_state, scenario.map_api, route_roadblock_ids)

coords, tl_data, speed_limit, lane_route = get_neighbor_vector_set_map(
    scenario.map_api,
    map_features=['LANE', 'LEFT_BOUNDARY', 'RIGHT_BOUNDARY', 'ROUTE_LANES'],
    point=ego_coords,
    radius=100.0,
    traffic_light_status_data=traffic_light_data,
)

vector_map = map_process(
    route_roadblock_ids,
    anchor_ego_state,
    coords,
    tl_data,
    speed_limit,
    lane_route,
    map_features=['LANE', 'LEFT_BOUNDARY', 'RIGHT_BOUNDARY', 'ROUTE_LANES'],
    max_elements={'LANE': cfg_dbg.lane_num, 'LEFT_BOUNDARY': cfg_dbg.lane_num, 'RIGHT_BOUNDARY': cfg_dbg.lane_num, 'ROUTE_LANES': cfg_dbg.route_num},
    max_points={'LANE': cfg_dbg.lane_len, 'LEFT_BOUNDARY': cfg_dbg.lane_len, 'RIGHT_BOUNDARY': cfg_dbg.lane_len, 'ROUTE_LANES': cfg_dbg.route_len},
)
print("vector_map keys:", sorted(vector_map.keys()))
for k, v in vector_map.items():
    if hasattr(v, "shape"):
        print(" ", k, v.shape, v.dtype)

# 6) assemble final dict (before tensor conversion)
data_np = {
    "neighbor_agents_past": neighbor_agents_past[:, -21:],
    "ego_current_state": np.array([0., 0., 1., 0., 0., 0., 0., 0., 0., 0.], dtype=np.float32),
    "static_objects": static_objects,
}
data_np.update(vector_map)

# 7) convert to tensors
inputs_dbg = convert_to_model_inputs(data_np, device="cpu")
print("\nconvert_to_model_inputs keys:", sorted(inputs_dbg.keys()))
for k, v in inputs_dbg.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: shape={tuple(v.shape)} dtype={v.dtype}")


## Tutorial: why the encoder/decoder tensor shapes look like this

### Encoder tokens (the `N` dimension)
In `diffusion_planner/model/module/encoder.py` the encoder builds a single token sequence by concatenating:

- **agents**: `config.agent_num`
- **static objects**: `config.static_objects_num`
- **lanes**: `config.lane_num`

So:

\[
N = agent\_num + static\_objects\_num + lane\_num
\]

This is the `token_num` used internally.

### Encoder output
The encoder produces:

- `encoder_outputs['encoding']`: shape **(B, N, hidden_dim)**

This is cross-attention context for the diffusion decoder.

### Decoder (diffusion) shapes
The decoder predicts future trajectories for **P** entities:

\[
P = 1 + predicted\_neighbor\_num
\]

- the `1` is the ego
- neighbors are the closest `predicted_neighbor_num` agents

At inference time it samples a trajectory of length `future_len` and each state is \(x,y,$\cos$,$\sin$\) = 4 dims.

So the final prediction is:

- `dec_out['prediction']`: shape **(B, P, future_len, 4)**

### Why `x,y,cos,sin`?
This avoids angle wraparound at \($\pm$ $\pi$\) and makes the model output smooth in heading.

### Check later:
- Add new map layers: update `DataProcessor._map_features` and extend `map_process`.
- Tokenization: adjust encoder `token_num` and the per-modality encoders?
- Improve agent selection: see `agent_past_process` (padding + filtering strategy).


```
Model input keys: ['ego_current_state', 'lanes', 'lanes_has_speed_limit', 'lanes_speed_limit', 'neighbor_agents_past', 'route_lanes', 'route_lanes_has_speed_limit', 'route_lanes_speed_limit', 'static_objects']
  neighbor_agents_past: (1, 32, 21, 11) torch.float32 cpu
  ego_current_state: (1, 10) torch.float32 cpu
  static_objects: (1, 5, 10) torch.float32 cpu
  lanes: (1, 70, 20, 12) torch.float32 cpu
  lanes_speed_limit: (1, 70, 1) torch.float32 cpu
  lanes_has_speed_limit: (1, 70, 1) torch.bool cpu
  route_lanes: (1, 25, 20, 12) torch.float32 cpu
  route_lanes_speed_limit: (1, 25, 1) torch.float32 cpu
  route_lanes_has_speed_limit: (1, 25, 1) torch.bool cpu

Encoder output keys: ['encoding']
  encoding: (1, 107, 192)

Decoder output keys: ['prediction']
  prediction: (1, 11, 80, 4)

prediction sample: tensor([[ 7.1128e+01, -5.2280e+00,  8.1615e-01,  2.7339e-02],
        [ 7.1644e+01, -5.0082e+00,  8.0531e-01,  1.0233e-03],
        [ 7.2167e+01, -4.7877e+00,  7.9777e-01, -1.9502e-02],
        [ 7.2704e+01, -4.5792e+00,  7.8902e-01, -4.3933e-02],
        [ 7.3236e+01, -4.3776e+00,  7.8086e-01, -5.7485e-02]])
```

```
anchor_ego_state [x,y,heading]: [ 6.64435983e+05  3.99718487e+06 -1.54833843e+00]
observation_buffer len: 21 type[0]: <class 'nuplan.planning.simulation.observation.observation_type.DetectionsTracks'>
raw neighbor frames: 21
raw neighbor frame[0] shape: (66, 8) (AgentInternalIndex.dim)
raw static_objects shape: (106, 5)
processed neighbor_agents_past shape: (32, 21, 11) (T, N, D or N,T,...) depending on function)
processed static_objects shape: (5, 10)
vector_map keys: ['lanes', 'lanes_has_speed_limit', 'lanes_speed_limit', 'route_lanes', 'route_lanes_has_speed_limit', 'route_lanes_speed_limit']
  lanes (70, 20, 12) float32
  lanes_speed_limit (70, 1) float32
  lanes_has_speed_limit (70, 1) bool
  route_lanes (25, 20, 12) float32
  route_lanes_speed_limit (25, 1) float32
  route_lanes_has_speed_limit (25, 1) bool

convert_to_model_inputs keys: ['ego_current_state', 'lanes', 'lanes_has_speed_limit', 'lanes_speed_limit', 'neighbor_agents_past', 'route_lanes', 'route_lanes_has_speed_limit', 'route_lanes_speed_limit', 'static_objects']
  neighbor_agents_past: shape=(1, 32, 21, 11) dtype=torch.float32
  ego_current_state: shape=(1, 10) dtype=torch.float32
  static_objects: shape=(1, 5, 10) dtype=torch.float32
  lanes: shape=(1, 70, 20, 12) dtype=torch.float32
  lanes_speed_limit: shape=(1, 70, 1) dtype=torch.float32
  lanes_has_speed_limit: shape=(1, 70, 1) dtype=torch.bool
  route_lanes: shape=(1, 25, 20, 12) dtype=torch.float32
  route_lanes_speed_limit: shape=(1, 25, 1) dtype=torch.float32
  route_lanes_has_speed_limit: shape=(1, 25, 1) dtype=torch.bool

```